## 1) BM25 Retriever


1) A probabilistic ranking function for sparse, keyword-based search.
2) Scores documents based on the frequency of query terms within them.
3) Incorporates two key normalization principles:

    a) Term Frequency Saturation: Prevents very long documents from dominating solely due to high term counts.
    
    b) Inverse Document Frequency: Penalizes terms that are common across the entire corpus, rewarding rare, specific terms.
4) Functions as a bag-of-words model, understanding syntax or word order.
5) Highly effective for term-matching precision and remains a strong, efficient baseline.


In [1]:
# Importing Necessary libaries.

from langchain_community.retrievers import BM25Retriever
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader


/Users/pranavk/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# import openparse

# parser = openparse.DocumentParser()
# parsed_basic_doc = parser.parse("/Users/pranavk/Desktop/HiWi/RAG/Efficient-and-Robust-Information_Retrieval-AES/pdfs/TR-03109-5_Testspezifikation_german.pdf")
# parsed_basic_doc



In [3]:
# for node in parsed_basic_doc.nodes:
#     print(node.text)
#     break

In [4]:
# from langchain.schema import Document
# documents = [Document(page_content=node.text) for node in parsed_basic_doc.nodes]
# documents[:4]

In [5]:
loader = PyPDFLoader("/Users/pranavk/Desktop/HiWi/RAG/Efficient-and-Robust-Information_Retrieval-AES/pdfs/Annex_PP_SMGW.pdf")
parsed_basic_doc = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 400,
    chunk_overlap  = 50,
)

docs_after_split = text_splitter.split_documents(parsed_basic_doc)

print(docs_after_split[1])

page_content='Bundesamt für Sicherheit in der Informationstechnik
PO Box 20 03 63
53133 Bonn
E-Mail: smartmeter@bsi.bund.de
Internet: https://www.bsi.bund.de
©  Federal Office for Information Security 2024' metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'author': 'Bundesamt für Sicherheit in der Informationstechnik', 'subject': '', 'title': 'Annex to Protection Profile for a Smart Meter Gateway (SMGW-PP) - Functional Package Multiple Grid Connection Points - Version 1.0 - 13.12.2024', 'source': '/Users/pranavk/Desktop/HiWi/RAG/Efficient-and-Robust-Information_Retrieval-AES/pdfs/Annex_PP_SMGW.pdf', 'total_pages': 12, 'page': 1, 'page_label': 'ii'}


In [6]:
from langchain_community.retrievers import BM25Retriever
from nltk.tokenize import word_tokenize
bm25_retriever = BM25Retriever.from_documents(docs_after_split,
                                            k=3,
                                            preprocess_func=word_tokenize)

def bm25_retrieve(query):
    bm25_retriever = BM25Retriever.from_documents(docs_after_split,
                                            k=3,
                                            preprocess_func=word_tokenize)
    result=bm25_retriever.invoke(query)
    for i,r in enumerate(result):
        print(f"Chunk {i}----------------------------------------------------------------------------------------------------------------------- \n{r.page_content} \n")
    

In [7]:
bm25_retrieve("1.7.1")

Chunk 0----------------------------------------------------------------------------------------------------------------------- 
1.7 Security functional requirements ............................................................................................ 4
1.7.1 FCS_COP.1/MTRCMAC ................................................................................................... 4
1.7.2 FTP_PRO.*/SYM ............................................................................................................. 4 

Chunk 1----------------------------------------------------------------------------------------------------------------------- 
1 Functional package: Multiple grid connection points
1.7 Security functional requirements
The following security functional requirements shall be adjusted from or added to the base PP if this functional
package is chosen.
1.7.1 FCS_COP.1/MTRCMAC
FCS_COP.1.1/MTRCMAC has to be replaced as follows:
FCS_COP.1.1/
MTRCMAC
The TSF shall perform [ MAC gener

In [8]:
bm25_retrieve("FPT_PHP.2.2")

Chunk 0----------------------------------------------------------------------------------------------------------------------- 
promise the TSF.
FPT_PHP.2.2 The TSF shall provide the capability to determine whether physical tampering with
the TSF's devices or TSF's elements has occurred.
FPT_PHP.2.3 For [all TSF devices/elements], the TSF shall monitor the devices and elements and
notify [the GWA, the consumer ] when physical tampering with the TSF's devices or
TSF's elements has occurred.
Hierarchical to: FPT_PHP.1 

Chunk 1----------------------------------------------------------------------------------------------------------------------- 
References
6  Federal Office for Information Security 

Chunk 2----------------------------------------------------------------------------------------------------------------------- 
List of Tables
List of Tables
1.1. Identification of functional package multiple grid connection points ............................................... 1
1.2. Adjus

## 2) Semantic Retrieval: with embeddings

1) Documents and queries are converted into vectors in the same high-dimensional space.

2) Retrieval is performed by finding the nearest document vectors to the query vector.

3) Similarity is measured using metrics like cosine similarity or dot product.

4) Excels at capturing semantic relationships and synonymy, going beyond literal keyword matches.


In [9]:
import torch
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

device=torch.device("mps")

huggingface_embeddings = HuggingFaceEmbeddings(
    model_name="jinaai/jina-embeddings-v3",  # alternatively use "sentence-transformers/all-MiniLM-l6-v2" for a light and faster experience.
    model_kwargs={'device':device,
    "trust_remote_code": True}, 
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = FAISS.from_documents(docs_after_split, huggingface_embeddings)
semantic_retriever=vectorstore.as_retriever()


/var/folders/lv/_v493kr51pz257bjfm4fz8580000gn/T/ipykernel_37504/3486509158.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  huggingface_embeddings = HuggingFaceEmbeddings(
/Users/pranavk/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:

def faiss_retrieve(query):
    relevant_documents = vectorstore.similarity_search(query)
    print(f'Total Documents retrieved: {len(relevant_documents)}')
    for i,doc in enumerate(relevant_documents):
        print(f"{i} doc")
        print(doc.page_content)
        print("\n")
        

In [11]:
faiss_retrieve(query="FPT_PHP.2.2")

Total Documents retrieved: 4
0 doc
1.7.3 FPT_PHP.2 (optional) ...................................................................................................... 4
1.7.4 Security functional requirements rationale ...................................................................... 4


1 doc
The text of FTP_PRO.*/SYM remains unchanged.
1.7.3 FPT_PHP.2 (optional)
If FPT_PHP.2 is included in the package to address parts of A.PhysicalProtection, it has to be included as fol-
lows:
1.7.3.1 FPT_PHP.2: Notification of physical attack
FPT_PHP.2.1 The TSF shall provide unambiguous detection of physical tampering that can com-
promise the TSF.


2 doc
Hierarchical to: FPT_PHP.1
Dependencies: FMT_MOF.1 unfulfilled, see Application Note below.
Application Note 2 When including FPT_PHP.2 in the ST, the ST author shall expand the table for
FPT_FLS.1 in [PP-0073] to add the detection of intrusion by the TOE as in FPT_FLS.1
and the reaction of the TOE. Further, the ST author shall add the event "

In [12]:
faiss_retrieve(query="1.7.1")


Total Documents retrieved: 4
0 doc
1.2 Package overview ................................................................................................................... 1
1.3 Definition of terms ................................................................................................................. 1


1 doc
1.4.2 Assumptions ................................................................................................................... 2
1.4.3 Attackers ........................................................................................................................ 2


2 doc
1.7.3 FPT_PHP.2 (optional) ...................................................................................................... 4
1.7.4 Security functional requirements rationale ...................................................................... 4


3 doc
1.4.4 Threats .......................................................................................................................

### 3) Hybrid Retrieval

1) Combines the strengths of both sparse (e.g., BM25) and dense (e.g., embedding-based) retrieval methods.
2) Aims to achieve both high recall (via semantic search) and high precision (via keyword search).
3) Common fusion methods include:

    a) Reciprocal Rank Fusion (RRF): Ranks documents based on the sum of their reciprocal ranks from each list, favoring documents that rank highly in both.
    
    b) Weighted Scoring: Combines the normalized scores from each method using a tunable weight.


In [13]:
from langchain.retrievers import EnsembleRetriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever], weights=[0.4, 0.6]
)

ensemble_retriever.invoke("1.7.1")

[Document(id='e5899b09-9e00-4edf-a54c-784e8946e2d0', metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'author': 'Bundesamt für Sicherheit in der Informationstechnik', 'subject': '', 'title': 'Annex to Protection Profile for a Smart Meter Gateway (SMGW-PP) - Functional Package Multiple Grid Connection Points - Version 1.0 - 13.12.2024', 'source': '/Users/pranavk/Desktop/HiWi/RAG/Efficient-and-Robust-Information_Retrieval-AES/pdfs/Annex_PP_SMGW.pdf', 'total_pages': 12, 'page': 3, 'page_label': 'iv'}, page_content='1.2 Package overview ................................................................................................................... 1\n1.3 Definition of terms ................................................................................................................. 1'),
 Document(id='dd7b5e45-0d37-4c30-9099-3f8906e2ab34', metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'author': 'Bundesamt für Sicherheit in der Informatio

In [14]:
def hybrid_retriver(query):
    ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever], weights=[0.6, 0.4])
    relevant_documents=ensemble_retriever.invoke(query)
    for i,doc in enumerate(relevant_documents):
        print(f"{i} doc")
        print(doc.page_content)
        print("\n")



In [15]:
hybrid_retriver("1.7.1")

0 doc
1.7 Security functional requirements ............................................................................................ 4
1.7.1 FCS_COP.1/MTRCMAC ................................................................................................... 4
1.7.2 FTP_PRO.*/SYM ............................................................................................................. 4


1 doc
1 Functional package: Multiple grid connection points
1.7 Security functional requirements
The following security functional requirements shall be adjusted from or added to the base PP if this functional
package is chosen.
1.7.1 FCS_COP.1/MTRCMAC
FCS_COP.1.1/MTRCMAC has to be replaced as follows:
FCS_COP.1.1/
MTRCMAC
The TSF shall perform [ MAC generation and verification for secure communication


2 doc
References
6  Federal Office for Information Security


3 doc
1.2 Package overview .........................................................................................................

In [16]:
hybrid_retriver("How attackers get access to")

0 doc
vulnerability assessment, that local attackers have at most proficient expertise and use at most specialized
equipment. However, the preparation of local attacks in terms of developing the attack path including tools
can be performed by attackers with full AVA_VAN.5 attack potential (i.e., multiple expert expertise and mul-


1 doc
definition.
Local attacker: An attacker that
• either has physical access to meter, TOE, a connection between these components, or local logical access
to any of the interfaces, who tries to disclose or alter assets while stored in the TOE or while transmitted
between external entities and the TOE;


2 doc
tiple bespoke equipment), as long as their execution is possible with the restrictions for local attackers men-
tioned above (also for all remote attacks (cf. following bullet point) the full AVA_VAN.5 attack potential has
to be assumed during the vulnerability assessment).
Please note that the local attacker includes authorized entities like consume

In [19]:
hybrid_retriver("What is defination of local attacker in [PP-0073]")

0 doc
to the environment where the TOE is located.
1.4.4 Threats
This package does not contain any additional threats. Note, however, that by adjusting the definition of the
local attacker, all threats with the threat agent being a local attacker address a wider range of attack vectors.
This concerns the following threats:
• T.DataModificationLocal
• T.TimeModification
• T.DisclosureLocal


1 doc
to the distance of the TOE to connected devices of bundled grid connection points.
A bundling of grid connection points with one TOE is permitted provided that the
grid connection points are connected to one grid node of the same voltage level.
1.4.3 Attackers
The definition of the local attacker in [PP-0073], Section 3.4 in the base PP shall be replaced by the following
definition.


2 doc
Sponsor This functional package inherits the sponsor of the base PP. Please refer to [PP-0073] Section 1.2.
Registration This functional package inherits the certification ID of the base PP. Please refer to

In [17]:
pwd

'/Users/pranavk/Desktop/HiWi/RAG/Efficient-and-Robust-Information_Retrieval-AES/local_experiments'

In [20]:

from src.llm_api import make_api_call

def ask_question(query):
    relevant_documents = vectorstore.similarity_search(query,k=3)
    context=relevant_documents[0].page_content +"/n"+relevant_documents[1].page_content

    prompt_template = f"""Use the following pieces of context to answer the question at the end. Please follow the following rules:
1. If you find the answer, write the answer in a concise way.
2. Answer in english.  

Context: {context}

Question: {query}

Helpful Answer:
"""
    out=make_api_call(prompt_template)
    out=out.split("Helpful Answer:")[-1].strip()

    return out

query="""What is defination of local attacker in [PP-0073]"""  
print(ask_question(query))

An attacker that has physical access to meter, TOE, a connection between these components, or local logical access to any of the interfaces.
